In [34]:
import gc
import torch

# 1. Delete previous model, trainer, and dataset objects if they exist
for obj_name in ['model', 'base_model', 'trainer', 'train_dataset', 'val_dataset', 'test_dataset']:
    if obj_name in globals():
        del globals()[obj_name]

# 2. Force Python garbage collection
gc.collect()

# 3. Clear PyTorch CUDA VRAM cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU Memory Cleared! Available VRAM: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")

# 4. Remove previous RAG context columns from DataFrames so new ones are generated fresh
for df_obj in [train_df, val_df, test_df]:
    if 'rag_context' in df_obj.columns:
        df_obj.drop(columns=['rag_context'], inplace=True)

print("Environment successfully reset for Contextual Dense RAG!")

GPU Memory Cleared! Available VRAM: 14.74 GB
Environment successfully reset for Contextual Dense RAG!


# Exploratory Data Analysis

In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [3]:
df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

print("=== DATA SHAPE & COLUMNS ===")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}\n")

print("=== MISSING VALUES ===")
print(df.isnull().sum())
print("\n=== SAMPLE HEAD ===")
print(df.head(2))

if 'answer' in df.columns:
    print("\n=== TARGET DISTRIBUTION ===")
    print(df['answer'].value_counts(dropna=False, normalize=True) * 100)

=== DATA SHAPE & COLUMNS ===
Shape: (2000, 8)
Columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']

=== MISSING VALUES ===
id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

=== SAMPLE HEAD ===
   id                                             prompt  \
0   1  Pick the best possible answer: What is Martin ...   
1   2        What is accelerator-based light-ion fusion?   

                                                   A  \
0  Martin Heidegger believes that humans exist wi...   
1  Accelerator-based light-ion fusion is a techni...   

                                                   B  \
0  Martin Heidegger believes that humans do not e...   
1  Accelerator-based light-ion fusion is a techni...   

                                                   C  \
0  Martin Heidegger does not believe in the exist...   
1  Accelerator-based light-ion fusion is a techni...   

                                                   

In [4]:
option_cols = ['A', 'B', 'C', 'D', 'E']
prompt_col = 'prompt'
answer_col = 'answer'

df['answer_col'] = df.apply(lambda row: row[row['answer']], axis=1)

if prompt_col:
    df['prompt_word_len'] = df[prompt_col].astype(str).apply(lambda x: len(x.split()))
if answer_col:
    df['answer_word_len'] = df['answer_col'].astype(str).apply(lambda x: len(x.split()))

for col in option_cols:
    df[f'{col}_word_len'] = df[col].astype(str).apply(lambda x: len(x.split()))

df['total_word_len'] = df[[c for c in df.columns if 'word_len' in c]].sum(axis=1)

print("=== LENGTH STATS (WORD COUNTS) ===")
print(df[[c for c in df.columns if 'word_len' in c]].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

=== LENGTH STATS (WORD COUNTS) ===
       prompt_word_len  answer_word_len   A_word_len   B_word_len  C_word_len  \
count       2000.00000      2000.000000  2000.000000  2000.000000  2000.00000   
mean          18.14650        28.659000    26.146000    26.516000    26.54950   
std            6.78189        18.331005    17.249651    18.653893    17.20987   
min            3.00000         1.000000     1.000000     1.000000     1.00000   
50%           17.00000        28.000000    22.000000    23.000000    24.00000   
90%           27.00000        51.000000    49.000000    49.000000    48.00000   
95%           31.00000        62.000000    62.000000    62.000000    60.00000   
99%           37.00000        84.000000    78.000000    84.000000    81.00000   
max           51.00000       105.000000    81.000000   118.000000    82.00000   

        D_word_len   E_word_len  total_word_len  
count  2000.000000  2000.000000     2000.000000  
mean     25.999500    26.187000      178.203500  
std 

In [5]:
for col in option_cols:
    df[f'{col}_char_len'] = df[col].astype(str).apply(len)

if 'answer' in df.columns:
    def get_correct_option_len(row):
        ans = str(row['answer']).strip()
        for col in option_cols:
            if ans == col:
                return len(str(row[col]))
        return None

    df['correct_option_char_len'] = df.apply(get_correct_option_len, axis=1)
    
    print("=== OPTION LENGTH COMPARISON ===")
    print(f"Mean Correct Option Length: {df['correct_option_char_len'].mean():.2f} chars")
    print("Mean Option Lengths by Column:")
    for col in option_cols:
        print(f"  {col}: {df[f'{col}_char_len'].mean():.2f} chars")

=== OPTION LENGTH COMPARISON ===
Mean Correct Option Length: 181.88 chars
Mean Option Lengths by Column:
  A: 164.09 chars
  B: 166.62 chars
  C: 167.23 chars
  D: 162.56 chars
  E: 164.23 chars


In [6]:
if prompt_col:
    exact_dup_prompts = df[prompt_col].duplicated().sum()
    print("=== DUPLICATE ANALYSIS ===")
    print(f"Exact Duplicate Prompts: {exact_dup_prompts} ({(exact_dup_prompts/len(df))*100:.2f}%)")

def check_duplicate_options(row):
    opts = [str(row[col]).strip() for col in option_cols]
    return len(opts) != len(set(opts))

df['has_duplicate_options'] = df.apply(check_duplicate_options, axis=1)
print(f"Questions with duplicate options: {df['has_duplicate_options'].sum()}")

=== DUPLICATE ANALYSIS ===
Exact Duplicate Prompts: 242 (12.10%)
Questions with duplicate options: 0


In [7]:
import re

def detect_patterns(text):
    text = str(text)
    has_latex = bool(re.search(r'\$|\\frac|\\sqrt|\\alpha|\\beta', text))
    has_code = bool(re.search(r'def\s+|class\s+|import\s+|for\s+.*in|#include', text))
    has_math_symbols = bool(re.search(r'[=+\-*/^<>≤≥≠±]', text))
    return pd.Series([has_latex, has_code, has_math_symbols], index=['has_latex', 'has_code', 'has_math_symbols'])

if prompt_col:
    pattern_df = df[prompt_col].apply(detect_patterns)
    print("=== DOMAIN & SYNTAX PATTERNS ===")
    print(pattern_df.mean() * 100)

=== DOMAIN & SYNTAX PATTERNS ===
has_latex            0.00
has_code             8.35
has_math_symbols    22.85
dtype: float64


# 1.) Roberta Base Model ( Pretrained )

In [35]:
import os
import wandb
import torch
from torch.utils.data import DataLoader
from dataclasses import dataclass
from typing import Optional, Union
from transformers import (
AutoTokenizer, 
AutoModelForMultipleChoice, 
TrainingArguments, 
Trainer,
PreTrainedTokenizerBase
)
from transformers.tokenization_utils_base import PaddingStrategy
from datasets import Dataset
from sklearn.model_selection import GroupKFold
from kaggle_secrets import UserSecretsClient

## Configuration and Preprocessing

In [36]:
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("WB_KEY")
wandb.login(key=secret_value_0)

wandb.init(
    project="23f2002267-t22026", 
    name="roberta-base-baseline",
    reinit="finish_previous",
    #config="{}",
    #notes="",
    #dir="./wandb_logs",
    #resume="allow, must, never",
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [37]:
MODEL_NAME = "roberta-base"
MAX_LEN = 256
option_cols = ['A', 'B', 'C', 'D', 'E']
label2id = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
df['label'] = df['answer'].map(label2id)

from datasets import Dataset
gkf = GroupKFold(n_splits=5)
df['fold'] = -1
for fold, (train_idx, val_idx) in enumerate(gkf.split(df, groups=df['prompt'])):
    df.loc[val_idx, 'fold'] = fold

train_df = df[df['fold'] != 0].reset_index(drop=True)
val_df = df[df['fold'] == 0].reset_index(drop=True)

### Tokenization

In [38]:
def preprocess_function(examples):
    first_sentences = [[f"Question: {prompt}"] * 5 for prompt in examples['prompt']] # copies for options
    second_sentences = [
        [f"Option {opt}: {examples[opt][i]}" for opt in option_cols] 
        for i in range(len(examples['prompt']))
    ]

    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])

    tokenized_examples = tokenizer(
        first_sentences, 
        second_sentences, 
        truncation=True, 
        max_length=MAX_LEN,
        padding=False
    )

    return {k: [v[i:i+5] for i in range(0, len(v), 5)] for k, v in tokenized_examples.items()}

In [39]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
train_dataset = Dataset.from_pandas(train_df).map(preprocess_function, batched=True)
val_dataset = Dataset.from_pandas(val_df).map(preprocess_function, batched=True)

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

## Data collator and compute_metrics

In [41]:
@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    # max_length: Optional[int] = None
    # pad_to_multiple_of: Optional[int] = None

    def __call__(self, features):
        if "label" in features[0]:
            label_name = "label"
            labels = [feature.pop(label_name) for feature in features]
        else:
            labels = None
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        
        flattened_features = [
            [{k: v[i] for k, v in feature.items()} for i in range(num_choices)]
            for feature in features
        ]
        flattened_features = sum(flattened_features, [])

        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            # max_length=self.max_length,
            # pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}

        if labels is not None:
            batch["labels"] = torch.tensor(labels, dtype=torch.long)
            
        return batch

# def compute_metrics(eval_predictions):
#     predictions, labels = eval_predictions
#     preds = np.argmax(predictions, axis=1)
#     acc = (preds == labels).mean()
#     return {"accuracy": acc}
def compute_metrics(eval_predictions):
    predictions, labels = eval_predictions
    top1_preds = np.argmax(predictions, axis=1)
    acc = (top1_preds == labels).mean()
    
    # Compute MAP@3
    top3_preds = np.argsort(predictions, axis=1)[:, ::-1][:, :3]
    scores = []
    for row, true_label in zip(top3_preds, labels):
        if true_label == row[0]: scores.append(1.0)
        elif true_label == row[1]: scores.append(0.5)
        elif true_label == row[2]: scores.append(1/3)
        else: scores.append(0.0)
    map3 = np.mean(scores)
    
    return {"accuracy": acc, "map@3": map3}

## Training

In [42]:
model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)
training_args = TrainingArguments(
        output_dir=f"./result",
        eval_strategy="epoch",
        save_strategy="no",
        save_only_model=True,
        learning_rate=1.5e-5,             # lowered lr for fine-grained convergence
        warmup_steps=100,
        max_grad_norm=1.0,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=2,    # Effective batch size = 8
        num_train_epochs=4,               
        label_smoothing_factor=0.05,      
        weight_decay=0.01,
        logging_steps=20,
        report_to="wandb",
        fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer, 
    data_collator=DataCollatorForMultipleChoice(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForMultipleChoice LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.weight               | MISSING    | 
roberta.pooler.dense.weight     | MISSING    | 
roberta.pooler.dense.bias       | MISSING    | 
classifier.bias                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Map@3
1,6.355364,3.090305,0.545000,0.696667
2,3.798920,1.457753,0.857500,0.906250
3,2.568869,1.029523,0.945000,0.966667
4,2.039032,0.874668,0.975000,0.985417


TrainOutput(global_step=400, training_loss=4.097309961318969, metrics={'train_runtime': 431.2966, 'train_samples_per_second': 14.839, 'train_steps_per_second': 0.927, 'total_flos': 1841370298977840.0, 'train_loss': 4.097309961318969, 'epoch': 4.0})

In [54]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Retrieve token from Kaggle Secrets
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_KEY")

# Log in programmatically with Write permissions
login(token=hf_token)

In [56]:
# 1. Save locally
model.save_pretrained("./roberta_mcq_model")
tokenizer.save_pretrained("./roberta_mcq_model")

# 2. Push to Hugging Face Hub (Replace with your HF username and model repo name)
# Make sure you have run `!huggingface-cli login` with your WRITE token first!
HF_REPO_NAME = "Pranjan007/roberta-mcq-solver"

model.push_to_hub(HF_REPO_NAME)
tokenizer.push_to_hub(HF_REPO_NAME)
print(f"Model successfully uploaded to Hugging Face Hub: {HF_REPO_NAME}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Model successfully uploaded to Hugging Face Hub: Pranjan007/roberta-mcq-solver


In [ ]:
wandb.finish()

## Inference and Submission

In [8]:
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample_sub = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

test_dataset = Dataset.from_pandas(test_df).map(preprocess_function, batched=True)
model_input_cols = [c for c in ["input_ids", "attention_mask", "token_type_ids"] if c in test_dataset.column_names]
test_dataset.set_format(type="python", columns=model_input_cols)

test_loader = DataLoader(
    test_dataset, 
    batch_size=8, 
    shuffle=False, 
    collate_fn=DataCollatorForMultipleChoice(tokenizer=tokenizer)
)

all_logits = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        all_logits.append(outputs.logits.cpu().numpy())

logits = np.concatenate(all_logits, axis=0)  # Shape: (500, 5)

id2label = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}
top3_indices = np.argsort(logits, axis=1)[:, ::-1][:, :3]

top3_predictions = [
    " ".join([id2label[idx] for idx in row]) 
    for row in top3_indices
]

id_col_name = sample_sub.columns[0]      # 'ID'
target_col_name = sample_sub.columns[1]  # 'Prediction'

submission_df = pd.DataFrame({
    id_col_name: test_df['id'],
    target_col_name: top3_predictions
})

submission_df.to_csv("submission.csv", index=False)

print("\n=== MAP@3 SUBMISSION HEAD ===")
print(submission_df.head(10))

print("\n=== TOP PREDICTION PATTERNS ===")
print(submission_df[target_col_name].value_counts().head(10))

Map:   0%|          | 0/500 [00:00<?, ? examples/s]


=== MAP@3 SUBMISSION HEAD ===
   ID Prediction
0   1      A D B
1   2      B D C
2   3      B D E
3   4      E C A
4   5      C A D
5   6      D B C
6   7      E B C
7   8      B A E
8   9      C D E
9  10      B C D

=== TOP PREDICTION PATTERNS ===
Prediction
D B C    31
B D C    26
C B D    26
E B D    24
C B A    24
E B C    23
C D B    22
B C A    20
A B C    20
B C D    19
Name: count, dtype: int64


# - Ensemble version:

## Initilization

In [9]:
MODEL_NAME = "roberta-base"
MAX_LEN = 256
NUM_FOLDS = 5
option_cols = ['A', 'B', 'C', 'D', 'E']
label2id = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
id2label = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
df['label'] = df['answer'].map(label2id)

gkf = GroupKFold(n_splits=NUM_FOLDS)
df['fold'] = -1
for fold, (train_idx, val_idx) in enumerate(gkf.split(df, groups=df['prompt'])):
    df.loc[val_idx, 'fold'] = fold

oof_predictions = np.zeros((len(df), 5))

## Preporcessing and Training

In [ ]:
wandb.init(
        project="23f2002267-t22026", 
        name=f"roberta-base-fold{fold}",
        reinit="finish_previous"
    )

In [13]:
FOLDS=2
for fold in range(FOLDS):
    print(f"\n========================== TRAINING FOLD {fold + 1}/{FOLDS} ==========================")

    fold_train = df[df['fold'] != fold].reset_index(drop=True)
    fold_val = df[df['fold'] == fold].reset_index(drop=True)

    train_dataset = Dataset.from_pandas(fold_train).map(preprocess_function, batched=True)
    val_dataset = Dataset.from_pandas(fold_val).map(preprocess_function, batched=True)

    model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

    training_args = TrainingArguments(
        output_dir=f"./results_fold{fold}",
        eval_strategy="epoch",
        save_strategy="no",               # FIXED: Disable intermediate checkpoint saving to prevent disk full error
        save_only_model=True,             # FIXED: Save only lightweight model weights
        learning_rate=2e-5,
        warmup_steps=50,
        max_grad_norm=1.0,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=2,
        num_train_epochs=3,
        weight_decay=0.01,
        logging_steps=20,
        report_to="none",
        fp16=torch.cuda.is_available()
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer, 
        data_collator=DataCollatorForMultipleChoice(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
    )

    trainer.train()
    
    # Save lightweight final model for this fold
    model.save_pretrained(f"./model_fold_{fold}")
    tokenizer.save_pretrained(f"./model_fold_{fold}")
    
    # Clean GPU memory between folds
    del model, trainer
    torch.cuda.empty_cache()
    wandb.finish()


========================== TRAINING FOLD 1/2 ==========================


Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForMultipleChoice LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.weight               | MISSING    | 
roberta.pooler.dense.weight     | MISSING    | 
roberta.pooler.dense.bias       | MISSING    | 
classifier.bias                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension

...

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForMultipleChoice LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.weight               | MISSING    | 
roberta.pooler.dense.weight     | MISSING    | 
roberta.pooler.dense.bias       | MISSING    | 
classifier.bias                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension

Epoch,Training Loss,Validation Loss,Accuracy
1,5.388861,2.401008,0.630000
2,2.004827,0.717448,0.892500
3,1.463453,0.468455,0.940000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Test data inference ans  Submission

In [14]:
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample_sub = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

test_dataset = Dataset.from_pandas(test_df).map(preprocess_function, batched=True)
model_input_cols = [c for c in ["input_ids", "attention_mask", "token_type_ids"] if c in test_dataset.column_names]
test_dataset.set_format(type="python", columns=model_input_cols)

test_loader = DataLoader(
    test_dataset, 
    batch_size=8, 
    shuffle=False, 
    collate_fn=DataCollatorForMultipleChoice(tokenizer=tokenizer)
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_fold_logits = []

for fold in range(FOLDS):
    print(f"Running inference with Fold {fold} model...")
    fold_model = AutoModelForMultipleChoice.from_pretrained(f"./model_fold_{fold}")
    fold_model.eval()
    fold_model.to(device)

    fold_logits = []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            
            outputs = fold_model(input_ids=input_ids, attention_mask=attention_mask)
            fold_logits.append(outputs.logits.cpu().numpy())

    all_fold_logits.append(np.concatenate(fold_logits, axis=0))
    del fold_model
    torch.cuda.empty_cache()

mean_logits = np.mean(all_fold_logits, axis=0)

top3_indices = np.argsort(mean_logits, axis=1)[:, ::-1][:, :3]
top3_predictions = [
    " ".join([id2label[idx] for idx in row]) 
    for row in top3_indices
]

id_col_name = sample_sub.columns[0]
target_col_name = sample_sub.columns[1]

submission_df = pd.DataFrame({
    id_col_name: test_df['id'],
    target_col_name: top3_predictions
})

submission_df.to_csv("submission.csv", index=False)

print("\n=== ENSEMBLED MAP@3 SUBMISSION HEAD ===")
print(submission_df.head(10))

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Running inference with Fold 0 model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Running inference with Fold 1 model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


=== ENSEMBLED MAP@3 SUBMISSION HEAD ===
   ID Prediction
0   1      A B D
1   2      B D C
2   3      B E D
3   4      E C D
4   5      C A D
5   6      D B A
6   7      E B C
7   8      B C E
8   9      C D E
9  10      B A D


# 2.) MLP-Scratch model

## Imports and Config

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import wandb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupKFold
from kaggle_secrets import UserSecretsClient

In [ ]:
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("WB_KEY")
wandb.login(key=secret_value_0)

wandb.init(
    project="23f2002267-t22026", 
    name="MLP-Scratch-v10",
    reinit="finish_previous",
    config={
        "MAX_FEATURES": 5000,
        "batch_size": 16,
        "hidden1": 256,
        "BatchNorm1d":256,
        "dropout": 0.3,
        "hidden2": 64,
        "BatchNorm1d": 64,
        "dropout": 0.2,
        "epochs": 10
    }
)

In [ ]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
label2id = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
id2label = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}
train_df['label'] = train_df['answer'].map(label2id)

gkf = GroupKFold(n_splits=5)
train_df['fold'] = -1
for fold, (train_idx, val_idx) in enumerate(gkf.split(train_df, groups=train_df['prompt'])):
    train_df.loc[val_idx, 'fold'] = fold

train_split = train_df[train_df['fold'] != 0].reset_index(drop=True)
val_split = train_df[train_df['fold'] == 0].reset_index(drop=True)

option_cols = ['A', 'B', 'C', 'D', 'E']

corpus = []
for _, row in train_split.iterrows():
    corpus.append(str(row['prompt']))
    for opt in option_cols:
        corpus.append(str(row[opt]))


MAX_FEATURES = 5000
vectorizer = TfidfVectorizer(max_features=MAX_FEATURES, stop_words='english')
vectorizer.fit(corpus)

## Functions

In [ ]:
def extract_tfidf_features(df):
    X = []
    y = []
    for _, row in df.iterrows():
        prompt_vec = vectorizer.transform([str(row['prompt'])]).toarray()[0]
        row_opts = []
        for opt in option_cols:
            opt_vec = vectorizer.transform([str(row[opt])]).toarray()[0]
            # # Absolute difference feature vector
            # diff_vec = np.abs(prompt_vec - opt_vec)
            # row_opts.append(diff_vec)
            # Concatenate Prompt TF-IDF vector and Option TF-IDF vector
            combined_vec = np.concatenate([prompt_vec, opt_vec])
            row_opts.append(combined_vec)
        X.append(row_opts)
        y.append(row['label'])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)

## Data Prep and loading

In [ ]:
print("Extracting TF-IDF Features for Train & Val...")
X_train, y_train = extract_tfidf_features(train_split)  # Shape: (1600, 5, 10000)
X_val, y_val = extract_tfidf_features(val_split)        # Shape: (400, 5, 10000)

class MCQDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        
    def __len__(self):
        return len(self.y)
        
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(MCQDataset(X_train, y_train), batch_size=16, shuffle=True) #16
val_loader = DataLoader(MCQDataset(X_val, y_val), batch_size=16, shuffle=False) #16

## MLP config

In [ ]:
class DeepMLPMCQSolver(nn.Module):
    def __init__(self, input_dim):
        super(DeepMLPMCQSolver, self).__init__()
        # 3-Layer Feedforward Neural Network
        self.scoring_head = nn.Sequential(
            nn.Linear(input_dim, 256), #256
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3), #0.3
            
            nn.Linear(256, 64), #256, 64
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(64, 1)  # 64, 1
        )

    def forward(self, x):
        # Input shape: (batch_size, 5, input_dim)
        batch_size, num_options, feature_dim = x.shape
        x_flat = x.view(-1, feature_dim)  # Flatten to (batch_size * 5, feature_dim)
        
        logits = self.scoring_head(x_flat) # Shape: (batch_size * 5, 1)
        logits = logits.view(batch_size, num_options)  # Unflatten to (batch_size, 5)
        return logits

## Training

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_dim = X_train.shape[2]
model = DeepMLPMCQSolver(input_dim=input_dim).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2) #weight_decay=1e-4

# 7. Training Loop
EPOCHS = 10 # 15
print(f"Starting Training for {EPOCHS} Epochs on {device}...")

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    correct = 0
    total = 0
    
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        logits = model(batch_X)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * len(batch_y)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == batch_y).sum().item()
        total += len(batch_y)
        
    train_acc = correct / total
    avg_train_loss = train_loss / total
    
    # Validation
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            logits = model(batch_X)
            loss = criterion(logits, batch_y)
            
            val_loss += loss.item() * len(batch_y)
            preds = torch.argmax(logits, dim=1)
            val_correct += (preds == batch_y).sum().item()
            val_total += len(batch_y)
            
    val_acc = val_correct / val_total
    avg_val_loss = val_loss / val_total
    
    # Log metrics to W&B
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "train_accuracy": train_acc,
        "val_loss": avg_val_loss,
        "val_accuracy": val_acc
    })
    
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}")

wandb.finish()

## Test Data Inference and Submission

In [ ]:
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample_sub = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

print(f"Test Set Shape: {test_df.shape}")

# 2. Feature Extraction for Test Set using the Trained TF-IDF Vectorizer
def extract_test_features(df):
    X = []
    for _, row in df.iterrows():
        prompt_vec = vectorizer.transform([str(row['prompt'])]).toarray()[0]
        row_opts = []
        for opt in option_cols:
            opt_vec = vectorizer.transform([str(row[opt])]).toarray()[0]
            # Concatenate Prompt TF-IDF vector and Option TF-IDF vector
            combined_vec = np.concatenate([prompt_vec, opt_vec])
            row_opts.append(combined_vec)
        X.append(row_opts)
    return np.array(X, dtype=np.float32)

print("Extracting TF-IDF Features for Test Set...")
X_test = extract_test_features(test_df)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)

# 3. Model Inference
model.eval()
with torch.no_grad():
    logits = model(X_test_tensor).cpu().numpy()  # Shape: (500, 5)

top3_indices = np.argsort(logits, axis=1)[:, ::-1][:, :3]

top3_predictions = [
    " ".join([id2label[idx] for idx in row]) 
    for row in top3_indices
]

# 5. Format and Save submission.csv
id_col_name = sample_sub.columns[0]      # 'ID'
target_col_name = sample_sub.columns[1]  # 'Prediction'

submission_df = pd.DataFrame({
    id_col_name: test_df['id'],
    target_col_name: top3_predictions
})

submission_df.to_csv("submission.csv", index=False)

print("\n=== SUBMISSION HEAD ===")
print(submission_df.head(10))

print("\n=== PREDICTION DISTRIBUTION ===")
print(submission_df[target_col_name].value_counts().head(10))

    MLP-Scratch-baseline : 0.60390
    MLP-Scratch-v1 : 0.61512
    MLP-Scratch-v3 : 0.70532
    MLP-Scratch-v4 : 0.70656
    MLP-Scratch-v5 : 0.71197
    MLP-Scratch-v6 : 0.70989
    MLP-Scratch-v7 : 0.70822
    MLP-Scratch-v8 : 0.70448
    MLP-Scratch-v9 : 0.70365
    MLP-Scratch-v10 : 0.70989

# 3.) DeBERTa-v3 with RAG & PEFT/LoRA

In [15]:
import os
import torch
import torch.nn as nn
import wandb
from dataclasses import dataclass
from typing import Optional, Union

from torch.utils.data import DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import pairwise_distances
from sklearn.model_selection import GroupKFold
from datasets import Dataset
from transformers.tokenization_utils_base import PaddingStrategy
from sentence_transformers import SentenceTransformer, CrossEncoder

from transformers import (
    AutoTokenizer, 
    AutoModelForMultipleChoice, 
    TrainingArguments, 
    Trainer,
    PreTrainedTokenizerBase
)
from peft import LoraConfig, get_peft_model, TaskType
from kaggle_secrets import UserSecretsClient

In [ ]:
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("WB_KEY")
wandb.login(key=secret_value_0)

wandb.init(
    project="23f2002267-t22026", 
    name="deberta-v3-rag-lora-base",
    reinit="finish_previous"
)

In [16]:
MODEL_NAME = "microsoft/deberta-v3-base"
RETRIEVER_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
MAX_LEN = 320
option_cols = ['A', 'B', 'C', 'D', 'E']
label2id = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
id2label = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

# 2. Data Loading & GroupKFold Split
df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
df['label'] = df['answer'].map(label2id)

gkf = GroupKFold(n_splits=5)
df['fold'] = -1
for fold, (train_idx, val_idx) in enumerate(gkf.split(df, groups=df['prompt'])):
    df.loc[val_idx, 'fold'] = fold

train_df = df[df['fold'] != 0].reset_index(drop=True)
val_df = df[df['fold'] == 0].reset_index(drop=True)

## Dense RAG Retriever using Pretrained Contextual Embeddings

In [ ]:
# print("Building RAG Document Retriever Corpus...")
# all_corpus = []
# for _,row in train_df.iterrows():
#     all_corpus.append(str(row[row['answer']]))
# retriever_vectorizer = TfidfVectorizer(max_features=10000, stop_words='english')
# retriever_vectorizer.fit(all_corpus)

# train_prompt_vecs = retriever_vectorizer.transform(all_corpus)

# def retrieve_context_rag(df):
#     """Simple RAG module: retrieves nearest contextual prompt from knowledge base."""
#     query_vecs = retriever_vectorizer.transform(df['prompt'].tolist())
#     dist_matrix = pairwise_distances(query_vecs, train_prompt_vecs, metric='cosine')
    
#     contexts = []
#     for i, row in enumerate(dist_matrix):
#         closest_idx = np.argsort(row)[0]
#         retrieved_text = train_df.iloc[closest_idx]['answer']
#         contexts.append(str(train_df[retrieved_text]))
#     return contexts

# train_df['rag_context'] = retrieve_context_rag(train_df)
# val_df['rag_context'] = retrieve_context_rag(val_df)

In [ ]:
# print("Loading Pretrained Contextual Sentence-Transformer Retriever...")
# retriever = SentenceTransformer(RETRIEVER_MODEL_NAME)

# all_corpus = []
# for _,row in train_df.iterrows():
#     all_corpus.append(str(row[row['answer']]))

# train_embeddings = retriever.encode(all_corpus, show_progress_bar=True, convert_to_tensor=True)

# def retrieve_dense_context(df):
#     query_prompts = df['prompt'].tolist()
#     query_embeddings = retriever.encode(query_prompts, show_progress_bar=False, convert_to_tensor=True)
    
#     similarity_matrix = torch.matmul(query_embeddings, train_embeddings.T).cpu().numpy()
    
#     retrieved_contexts = []
#     for i, row in enumerate(similarity_matrix):
#         top_idx = np.argsort(row)[-1]
#         retrieved_text = train_df.iloc[top_idx]['prompt']
#         retrieved_contexts.append(retrieved_text)
        
#     return retrieved_contexts

# train_df['rag_context'] = retrieve_dense_context(train_df)
# val_df['rag_context'] = retrieve_dense_context(val_df)

In [17]:
print("Loading Stage 1 Retriever & Stage 2 Reranker Models...")
retriever = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

all_corpus = []
for _,row in train_df.iterrows():
    all_corpus.append(str(row[row['answer']]))

print("Encoding Knowledge Base into Vector Index...")
corpus_embeddings = retriever.encode(all_corpus, convert_to_tensor=True, show_progress_bar=True)

def retrieve_and_rerank_contexts(queries, is_train=False, top_n_retrieve=15, top_k_rerank=1):
    
    query_embeddings = retriever.encode(queries, convert_to_tensor=True, show_progress_bar=False)
    
    # Compute Cosine Similarity Matrix (Num_Queries x Num_Corpus)
    sim_matrix = torch.matmul(query_embeddings, corpus_embeddings.T).cpu().numpy()
    
    final_contexts = []
    
    for i, query in enumerate(queries):
        
        candidate_indices = np.argsort(sim_matrix[i])[-top_n_retrieve:][::-1]
            
        candidate_passages = [all_corpus[idx] for idx in candidate_indices]
        
        cross_inp = [[query, passage] for passage in candidate_passages]
        rerank_scores = reranker.predict(cross_inp)
        
        reranked_indices = np.argsort(rerank_scores)[::-1][:top_k_rerank]
        
        selected_passages = [candidate_passages[idx] for idx in reranked_indices]
        combined_context = " | ".join(selected_passages)
        final_contexts.append(combined_context)
        
    return final_contexts

print("Running Two-Stage RAG (Retrieve 10 -> Rerank Top 2)...")
train_df['rag_context'] = retrieve_and_rerank_contexts(train_df['prompt'].tolist(), is_train=True)
val_df['rag_context'] = retrieve_and_rerank_contexts(val_df['prompt'].tolist(), is_train=False)

Loading Stage 1 Retriever & Stage 2 Reranker Models...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Encoding Knowledge Base into Vector Index...


Batches:   0%|          | 0/50 [00:00<?, ?it/s]

Running Two-Stage RAG (Retrieve 10 -> Rerank Top 2)...


## Tokenization & Data Formatting

In [18]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

train_dataset = Dataset.from_pandas(train_df).map(preprocess_function, batched=True)
val_dataset = Dataset.from_pandas(val_df).map(preprocess_function, batched=True)

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

## Data Collator & MAP@3 Metric Function

In [26]:
@dataclass
class RobustDataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True

    def __call__(self, features):
        labels = [feature.pop("label") for feature in features]
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        
        model_inputs = ["input_ids", "attention_mask", "token_type_ids"]
        flattened_features = [
            [{k: feature[k][i] for k in feature if k in model_inputs} for i in range(num_choices)]
            for feature in features
        ]
        flattened_features = sum(flattened_features, [])

        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            return_tensors="pt",
        )

        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        batch["labels"] = torch.tensor(labels, dtype=torch.long)
        return batch

def compute_metrics(eval_predictions):
    predictions, labels = eval_predictions
    top1_preds = np.argmax(predictions, axis=1)
    acc = (top1_preds == labels).mean()
    
    # Compute MAP@3
    top3_preds = np.argsort(predictions, axis=1)[:, ::-1][:, :3]
    scores = []
    for row, true_label in zip(top3_preds, labels):
        if true_label == row[0]: scores.append(1.0)
        elif true_label == row[1]: scores.append(0.5)
        elif true_label == row[2]: scores.append(1/3)
        else: scores.append(0.0)
    map3 = np.mean(scores)
    
    return {"accuracy": acc, "map@3": map3}

## Model Initialization & LoRA PEFT Injector

In [20]:
print("Loading Base DeBERTa-v3 Model...")
base_model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

# Configure LoRA Parameters
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,                         
    lora_alpha=32,               
    lora_dropout=0.1,
    target_modules=["query_proj", "value_proj"],
    bias="none"
)

model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters()

Loading Base DeBERTa-v3 Model...


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight              

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

trainable params: 590,593 || all params: 185,013,506 || trainable%: 0.3192


## Training

In [22]:
training_args = TrainingArguments(
    output_dir="./results_deberta_rag_lora",
    eval_strategy="epoch",
    save_strategy="epoch",
    metric_for_best_model="map@3",
    greater_is_better=True,
    load_best_model_at_end=True,
    learning_rate=1e-4,           # Higher LR suitable for LoRA adapters
    warmup_steps=50,
    max_grad_norm=1.0,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=4,
    weight_decay=0.01,
    logging_steps=20,
    report_to="none",
    fp16=False,
    bf16=torch.cuda.is_bf16_supported()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer, 
    data_collator=RobustDataCollatorForMultipleChoice(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

print("Starting DeBERTa-v3 RAG + LoRA Training...")
trainer.train()

# Save final LoRA adapter weights
model.save_pretrained("./deberta_v3_rag_lora_final")
tokenizer.save_pretrained("./deberta_v3_rag_lora_final")
wandb.finish()

Starting DeBERTa-v3 RAG + LoRA Training...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Map@3
1,6.383789,3.117188,0.342500,0.521667
2,6.169531,2.902344,0.412500,0.595000
3,5.990332,2.718750,0.485000,0.640417
4,5.831641,2.662109,0.517500,0.663333


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


## Inference and Submission

In [33]:
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

# exact_match_lookup = dict(zip(train_df['prompt'].str.strip(), train_df['answer']))

print("Generating RAG Contexts for Test Set...")
test_df['rag_context'] = retrieve_and_rerank_contexts(test_df['prompt'].to_list())
# test_df['rag_context'] = retrieve_context_rag(test_df)

# dummy labels
test_df['label'] = 0 

print("Tokenizing Test Data...")
test_dataset = Dataset.from_pandas(test_df).map(preprocess_function, batched=True)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=RobustDataCollatorForMultipleChoice(tokenizer=tokenizer)
)

print("Running DeBERTa-v3 RAG + LoRA Model Inference on Test Set...")
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

all_logits = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        
        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits.cpu().numpy()
        all_logits.append(logits)

test_logits = np.concatenate(all_logits, axis=0)

option_list = ['A', 'B', 'C', 'D', 'E']
final_predictions = []

top3_indices = np.argsort(test_logits, axis=1)[:, ::-1][:, :3]
cou = 0 
for idx, row in test_df.iterrows():
    clean_prompt = str(row['prompt']).strip()
        # if clean_prompt in exact_match_lookup:
    #     true_ans = exact_match_lookup[clean_prompt]
    #     # Put exact true answer first, followed by top 2 model guesses
    #     other_choices = [c for c in option_list if c != true_ans]
    #     top_str = f"{true_ans} {other_choices[0]} {other_choices[1]}"
    #     cou+=1
    # else:
    #     # Standard Model Top-3 Predictions
    row_preds = top3_indices[idx]
    top_str = f"{option_list[row_preds[0]]} {option_list[row_preds[1]]} {option_list[row_preds[2]]}"
        
    final_predictions.append(top_str)
print(cou)
# 6. Format & Save submission.csv
submission_df = pd.DataFrame({
    'id': test_df['id'],
    'prediction': final_predictions
})

submission_df.to_csv("submission.csv", index=False)
print("SUCCESS: submission.csv successfully generated and saved!")
print("\nFirst 5 Rows of Submission:")
print(submission_df.head())

Generating RAG Contexts for Test Set...
Tokenizing Test Data...


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Running DeBERTa-v3 RAG + LoRA Model Inference on Test Set...
0
SUCCESS: submission.csv successfully generated and saved!

First 5 Rows of Submission:
   id prediction
0   1      C B A
1   2      E B C
2   3      E B D
3   4      E C A
4   5      D C A
